# Engineering Drawing Tutor
**Gemma 4 Good Hackathon** | Theme: Education in Underserved Communities

Fine-tuned Gemma 4 E4B generating manufacturing checklists, explaining tolerances,
and flagging AS1100 compliance issues. Offline on T4 / CPU via GGUF.

In [ ]:
# Cell 1 — install and clone (training data is inside the repo)
import subprocess, sys, os
subprocess.run(["pip", "install", "unsloth[colab-new]", "trl", "datasets", "-q"], check=False)
subprocess.run(["git", "clone", "https://github.com/govindrathore27/gemma4-engineering-diagrams.git",
                "/kaggle/working/repo"], capture_output=True)
subprocess.run(["git", "-C", "/kaggle/working/repo", "pull"], capture_output=True)
sys.path.insert(0, "/kaggle/working/repo")

TRAIN_JSONL = "/kaggle/working/repo/project3_drawing_tutor/data/train.jsonl"
EVAL_JSONL  = "/kaggle/working/repo/project3_drawing_tutor/data/eval.jsonl"

n_train = sum(1 for _ in open(TRAIN_JSONL))
n_eval  = sum(1 for _ in open(EVAL_JSONL))
print(f"Training data ready: {n_train} train rows, {n_eval} eval rows")

In [ ]:
# Cell 2 — fine-tune Gemma 4 E4B with LoRA
os.environ["WANDB_DISABLED"] = "true"
from shared.model.train import TrainConfig, train

cfg = TrainConfig(
    project="drawing",
    data_path=TRAIN_JSONL,
    output_dir="/kaggle/working/drawing_adapter",
    epochs=3,
    batch_size=2,
    grad_accum=8,
)
train(cfg)
print("Training complete!")

In [ ]:
# Cell 3 — evaluate
import json
from pathlib import Path
from shared.model.inference import load_model, run
from shared.eval.metrics import evaluate_batch

model, tokenizer = load_model("/kaggle/working/drawing_adapter")
eval_rows = [json.loads(l) for l in open(EVAL_JSONL, encoding="utf-8")][:100]
predictions = [run(model, tokenizer, r["instruction"], r["input"]) for r in eval_rows]
expected    = [r["output"] for r in eval_rows]
em = evaluate_batch(predictions, expected, metric="exact_match")
f1 = evaluate_batch(predictions, expected, metric="f1")
print(f"Exact Match: {em['mean']:.3f}  |  Token F1: {f1['mean']:.3f}")
Path("/kaggle/working/eval_results.txt").write_text(
    f"Exact Match: {em['mean']:.3f}\nToken F1: {f1['mean']:.3f}\nSamples: {len(eval_rows)}\n")

In [ ]:
# Cell 4 — GGUF export
from shared.model.quantize import export_gguf
export_gguf(
    adapter_dir="/kaggle/working/drawing_adapter",
    output_path="/kaggle/working/drawing_q4km.gguf",
    quant_type="q4_k_m"
)
print("GGUF export complete")

In [ ]:
# Cell 5 — demo
ctx = "Shaft assembly, AS1100. Diameter 25.00mm ±0.05, keyway 6x6mm, length 120mm"
print("=== Engineering Drawing Tutor Demo ===\n")
for q in [
    "List the critical dimensions for a shaft drawing.",
    "Create a manufacturing checklist for this engineering drawing.",
    "How do you verify a drawing conforms to AS1100 standard?",
    "How do you read a tolerance of 25.00 ± 0.05?",
]:
    print(f"Q: {q}\nA: {run(model, tokenizer, q, ctx)}\n")

## Impact

50M+ machinist apprentices in developing countries receive AS1100 drawings but lack
access to instructors for GD&T, tolerances, and compliance. This model is an offline
pocket tutor running via llama.cpp on any workshop laptop.

**Training data:** 1,200 QA pairs — 24 gold AS1100 examples + synthetic GD&T templates.